In [11]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Border, Side, Alignment
from openpyxl.utils.dataframe import dataframe_to_rows
from datetime import datetime
import glob

# Step 1: Define filenames (fixing the matched and unmatched references)
vm_analysis_file = glob.glob("*VM Analysis*.xlsx")[0]
deduplication_file = glob.glob("*Deduplication*.xlsx")[0]
output_refined_file = glob.glob("*NAICS Refined*.xlsx")[0]
matched_export_file = glob.glob("teal_iq*.csv")[0]  # Read as CSV for matched files
unmatched_export_file = glob.glob("unmatched*.csv")[0]  # Read as CSV for unmatched files

# Step 2: Read Data from VM Analysis
vm_vendor_master = pd.read_excel(vm_analysis_file, sheet_name="Vendor Master")
vm_summary = pd.read_excel(vm_analysis_file, sheet_name="Summary")
vm_spend_summary = pd.read_excel(vm_analysis_file, sheet_name="Spend Summary")  # Spend Summary tab

# Step 3: Read Data from Deduplication
dedup_summary = pd.read_excel(deduplication_file, sheet_name="Duplicate SUMMARY")

# Step 4: Read Data from matched_export and unmatched_export CSV files
matched_export = pd.read_csv(matched_export_file, low_memory=False)
unmatched_export = pd.read_csv(unmatched_export_file, low_memory=False)

# Step 5: Read Data from Output Refined
output_refined_naics_list = pd.read_excel(output_refined_file, sheet_name="NAICS List")
output_refined_naics_codes = pd.read_excel(output_refined_file, sheet_name="NAICS Codes")

# Step 6: Define Variables

# Calculate Total Suppliers (from the VM Analysis "Vendor Master" tab)
total_suppliers = len(vm_vendor_master["internal supplier id"])

# Calculate Total Uniques (from the VM Analysis "Summary" tab, where "GRAND TOTAL" appears)
total_uniques = vm_summary.loc[vm_summary.iloc[:, 0] == "GRAND TOTAL"].iloc[0, 2]

# Deduplication counts from "Duplicate SUMMARY" tab
hard_duplicates = dedup_summary.loc[dedup_summary.iloc[:, 0] == "Hard Duplicate Rows Removed"].iloc[0, 1]
id_duplicates = dedup_summary.loc[dedup_summary.iloc[:, 0] == "ID Duplicates"].iloc[0, 1]
name_duplicates = dedup_summary.loc[dedup_summary.iloc[:, 0] == "Name Duplicates"].iloc[0, 1]
address_duplicates = dedup_summary.loc[dedup_summary.iloc[:, 0] == "Address Duplicates"].iloc[0, 1]

# Matched and Unmatched Supplier counts
matched_suppliers = matched_export["internal_supplier_id_or_vendor_number"].nunique()
unmatched_suppliers = unmatched_export["internal_supplier_id"].nunique()

# Code breakdown (A, P, G, C, I, T)
a_employee = vm_summary.loc[vm_summary.iloc[:, 0] == "A"].iloc[0, 2]
p_person = vm_summary.loc[vm_summary.iloc[:, 0] == "P"].iloc[0, 2]
g_govt = vm_summary.loc[vm_summary.iloc[:, 0] == "G"].iloc[0, 2]
c_college = vm_summary.loc[vm_summary.iloc[:, 0] == "C"].iloc[0, 2]
i_nonprofit = vm_summary.loc[vm_summary.iloc[:, 0] == "I"].iloc[0, 2]
t_trust = vm_summary.loc[vm_summary.iloc[:, 0] == "T"].iloc[0, 2]

# Total NAICS and Unique Suppliers with NAICS
total_naics = len(output_refined_naics_list["NAICS"])
unique_suppliers_with_naics = len(output_refined_naics_codes["Internal Supplier ID"])

# Define the category list for the summary
category_list = [
    "Total Suppliers",
    "Total Uniques",
    "Duplicate Counts",  # No actual count, just a label
    "Hard Duplicates",
    "ID Duplicates",
    "Name Duplicates",
    "Address Duplicates",
    "Matched vs Unmatched",  # No actual count, just a label
    "Matched Suppliers",
    "Unmatched Suppliers",
    "Code Summary",  # No actual count, just a label
    "A (Employee)",
    "P (Person)",
    "G (Govt)",
    "C (College)",
    "I (Nonprofit)",
    "T (Trust/Estate)",
    "NAICS Summary",  # No actual count, just a label
    "Total NAICS",
    "Unique Suppliers with NAICS"
]

# Step 7: Calculate Spend and Spend Percentage
total_spend = vm_spend_summary['aggregated spend'].sum()

# Spend for Matched Suppliers (adjusted to reference matched_export correctly)
matched_spend = vm_spend_summary.loc[
    vm_spend_summary["internal supplier id"].isin(matched_export["internal_supplier_id_or_vendor_number"]),
    "aggregated spend"
].sum()

# Spend for Unmatched Suppliers (adjusted to reference unmatched_export correctly)
unmatched_suppliers_count = total_uniques - matched_suppliers  # Count for Unmatched Suppliers
matched_suppliers_percentage = (matched_suppliers / total_uniques) if total_uniques else 0
unmatched_suppliers_percentage = 1 - matched_suppliers_percentage  # Ensure percentages add up to 100%
unmatched_spend = total_spend - matched_spend  # Spend for Unmatched Suppliers
matched_spend_percentage = (matched_spend / total_spend) if total_spend else 0
unmatched_spend_percentage = 1 - matched_spend_percentage  # Ensure spend percentages add up to 100%

# Spend for code categories in Vendor Master (A, P, G, C, I, T)
def calculate_spend_by_code(code):
    return vm_spend_summary.loc[
        vm_spend_summary["internal supplier id"].isin(vm_vendor_master.loc[vm_vendor_master["code"] == code, "internal supplier id"]),
        "aggregated spend"
    ].sum()

spend_a = calculate_spend_by_code("A")
spend_p = calculate_spend_by_code("P")
spend_g = calculate_spend_by_code("G")
spend_c = calculate_spend_by_code("C")
spend_i = calculate_spend_by_code("I")
spend_t = calculate_spend_by_code("T")

# Spend for Unique Suppliers with NAICS
unique_naics_spend = vm_spend_summary.loc[
    vm_spend_summary["internal supplier id"].isin(output_refined_naics_codes["Internal Supplier ID"]),
    "aggregated spend"
].sum()

# Summary Data for Spend and Spend Percentage
summary_data = {
    "Category": category_list,
    "Count": [
        total_suppliers,  # Total Suppliers
        total_uniques,  # Total Uniques
        None,  # "Duplicate Counts" (No Count logic, just a label)
        hard_duplicates,  # Hard Duplicates
        id_duplicates,  # ID Duplicates
        name_duplicates,  # Name Duplicates
        address_duplicates,  # Address Duplicates
        None,  # "Matched vs Unmatched" (No Count logic, just a label)
        matched_suppliers,  # Matched Suppliers
        unmatched_suppliers_count,  # Unmatched Suppliers (New Logic)
        None,  # "Code Summary" (No Count logic, just a label)
        a_employee,  # A (employee)
        p_person,  # P (person)
        g_govt,  # G (govt)
        c_college,  # C (college)
        i_nonprofit,  # I (nonprofit)
        t_trust,  # T (trust/estate)
        None,  # "NAICS Summary" (No Count logic, just a label)
        total_naics,  # Total NAICS
        unique_suppliers_with_naics  # Unique Suppliers with NAICS
    ],
    "Percentage": [
        "-",  # Total Suppliers
        "-",  # Total Uniques
        None,  # "Duplicate Counts" (No logic, just a label)
        hard_duplicates / total_suppliers if total_suppliers and hard_duplicates else "-",  # Hard Duplicates %
        id_duplicates / total_suppliers if total_suppliers and id_duplicates else "-",  # ID Duplicates %
        name_duplicates / total_suppliers if total_suppliers and name_duplicates else "-",  # Name Duplicates %
        address_duplicates / total_suppliers if total_suppliers and address_duplicates else "-",  # Address Duplicates %
        None,  # "Matched vs Unmatched" (No logic, just a label)
        matched_suppliers_percentage,  # Matched Suppliers %
        unmatched_suppliers_percentage,  # Unmatched Suppliers Percentage (New Logic)
        None,  # "Code Summary" (No logic, just a label)
        a_employee / total_uniques if total_uniques else "0",  # A (employee) %
        p_person / total_uniques if total_uniques else "0",  # P (person) %
        g_govt / total_uniques if total_uniques else "0",  # G (govt) %
        c_college / total_uniques if total_uniques else "0",  # C (college) %
        i_nonprofit / total_uniques if total_uniques else "0",  # I (nonprofit) %
        t_trust / total_uniques if total_uniques else "0",  # T (trust/estate) %
        None,  # "NAICS Summary" (No logic, just a label)
        "-",  # Total NAICS
        unique_suppliers_with_naics / total_uniques if total_uniques else "0",  # Unique Suppliers with NAICS %
    ],
    "Spend": [
        total_spend,  # Total Suppliers Spend
        total_spend,  # Total Uniques Spend
        "n/a",  # "Duplicate Counts" Spend
        "n/a",  # Hard Duplicates Spend
        "n/a",  # ID Duplicates Spend
        "n/a",  # Name Duplicates Spend
        "n/a",  # Address Duplicates Spend
        None,  # "Matched vs Unmatched" Spend
        matched_spend,  # Matched Suppliers Spend
        unmatched_spend,  # Unmatched Suppliers Spend (New Logic)
        None,  # "Code Summary" Spend
        spend_a,  # A (employee) Spend
        spend_p,  # P (person) Spend
        spend_g,  # G (govt) Spend
        spend_c,  # C (college) Spend
        spend_i,  # I (nonprofit) Spend
        spend_t,  # T (trust/estate) Spend
        None,  # "NAICS Summary" Spend
        "n/a",  # Total NAICS Spend
        unique_naics_spend  # Unique Suppliers with NAICS Spend
    ],
    "Spend Percentage": [
        "-",  # Total Suppliers Spend Percentage
        "-",  # Total Uniques Spend Percentage
        "n/a",  # "Duplicate Counts" Spend Percentage
        "n/a",  # Hard Duplicates Spend Percentage
        "n/a",  # ID Duplicates Spend Percentage
        "n/a",  # Name Duplicates Spend Percentage
        "n/a",  # Address Duplicates Spend Percentage
        None,  # "Matched vs Unmatched" Spend Percentage
        matched_spend_percentage,  # Matched Suppliers Spend Percentage
        unmatched_spend_percentage,  # Unmatched Suppliers Spend Percentage (New Logic)
        None,  # "Code Summary" Spend Percentage
        spend_a / total_spend if total_spend else "0",  # A (employee) Spend Percentage
        spend_p / total_spend if total_spend else "0",  # P (person) Spend Percentage
        spend_g / total_spend if total_spend else "0",  # G (govt) Spend Percentage
        spend_c / total_spend if total_spend else "0",  # C (college) Spend Percentage
        spend_i / total_spend if total_spend else "0",  # I (nonprofit) Spend Percentage
        spend_t / total_spend if total_spend else "0",  # T (trust/estate) Spend Percentage
        None,  # "NAICS Summary" Spend Percentage
        "-",  # Total NAICS Spend Percentage
        unique_naics_spend / total_spend if total_spend else "0",  # Unique Suppliers with NAICS Spend Percentage
    ]
}

# Convert to DataFrame
summary_df = pd.DataFrame(summary_data)

# Step 8: Create the Output File
today_date = datetime.today().strftime('%Y-%m-%d')
output_filename = f"Step 4_SDP Teal iQ Summary - {today_date}.xlsx"

# Create Excel Workbook and Add Data
wb = Workbook()
ws = wb.active
ws.title = "Summary"

# Write the DataFrame to Excel
for row in dataframe_to_rows(summary_df, index=False, header=True):
    ws.append(row)

# Step 9: Set column widths
ws.column_dimensions['A'].width = 30  # Category column
ws.column_dimensions['B'].width = 10  # Count column
ws.column_dimensions['C'].width = 10  # Percentage column
ws.column_dimensions['D'].width = 15  # Spend column
ws.column_dimensions['E'].width = 15  # Spend Percentage column

# Step 10: Apply teal highlight to header row
teal_fill = PatternFill(start_color="008080", end_color="008080", fill_type="solid")

for cell in ws[1]:
    cell.fill = teal_fill

# Step 11: Merge cells and apply color formatting to specific rows
highlights = {
    "Duplicate Counts": "FFA500",  # Orange
    "Matched vs Unmatched": "ADD8E6",  # Light Blue
    "Code Summary": "FFFF00",  # Yellow
    "NAICS Summary": "00FF00"  # Green
}

for index, row in enumerate(ws.iter_rows(min_row=2, max_row=len(category_list)+1), start=2):
    category_value = row[0].value
    if category_value in highlights:
        ws.merge_cells(f"A{index}:E{index}")  # Merge Category, Count, Percentage, Spend, Spend Percentage
        merged_cell = ws.cell(row=index, column=1)
        merged_cell.fill = PatternFill(start_color=highlights[category_value], end_color=highlights[category_value], fill_type="solid")

# Step 12: Increase font size for all cells
for row in ws.iter_rows():
    for cell in row:
        cell.font = Font(size=12)

# Step 13: Apply thin box outline to the specified ranges
thin_border = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))

# Define the ranges for box outlines
ranges = ["A1:E3", "A4:E8", "A9:E11", "A12:E18", "A19:E21"]

# Apply border to each range
for range_cells in ranges:
    for row in ws[range_cells]:
        for cell in row:
            cell.border = thin_border

# Step 14: Apply bold font to specific Category rows ("Duplicate Counts", "Matched vs Unmatched", "Code Summary", "NAICS Summary")
bold_categories = ["Duplicate Counts", "Matched vs Unmatched", "Code Summary", "NAICS Summary"]

for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=1):
    if row[0].value in bold_categories:
        row[0].font = Font(size=12, bold=True)  # Apply bold font

# Step 15: Apply white font color for headers
white_font = Font(size=12, bold=True, color="FFFFFF")
for cell in ws[1]:
    cell.font = white_font

# Step 16: Apply percentage formatting to Percentage and Spend Percentage columns and center "-" values
for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=2, max_col=5):  # Loop over the Count, Percentage, Spend, and Spend Percentage columns
    for cell in row:
        if isinstance(cell.value, (int, float)):  # Apply formatting only to numeric values
            if cell.column == 3 or cell.column == 5:  # Percentage or Spend Percentage columns
                cell.number_format = '0.00%'  # Apply percentage formatting
            elif cell.column == 2:  # Count column
                cell.number_format = '#,##0'  # Apply number format with commas to Count column
            elif cell.column == 4:  # Spend column
                cell.number_format = '#,##0.00'  # Apply number format with commas and two decimal places to Spend column
        elif cell.value == "-":  # Center align cells with "-" input
            cell.alignment = Alignment(horizontal='center')

# Step 17: Save the Excel file
wb.save(output_filename)
print(f"File saved as: {output_filename}")


File saved as: Step 4_SDP Teal iQ Summary - 2024-10-17.xlsx
